![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Build the Core

*Project 3 — AI Product: From Model to Product*

**Student:** Gabriel Gomes

**Track:** LLM

**Data:** Criterion Collection

---

Build your solution, compare the design choices with evidence, and choose your final version.

> **Done when:** you have an end-to-end result that beats (or honestly fails to beat) the baseline, and you can say why you chose your final version.

---
## Setup

In [1]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/
sys.path.append("../src")      # the src scripts import each other by bare name

import json

import pandas as pd

from src.first_look import load_tables
from src.eval_checks import prepare_catalogue
from src.prepare_documents import format_citation, load_chunks
from src.functions import *

pd.set_option("display.max_colwidth", 200)

---
## 1. The pipeline

**Retrieval.** All 27,576 chunks are embedded locally with `all-MiniLM-L6-v2` and stored in a Chroma index in `models/chroma`.

**Metadata filters.** Filters are part of the pipeline from the start. A first call to the LLM extracts any hard constraints stated in the question (years, genre, country, director), and the top `k` chunks are retrieved only from the chunks that meet them. A question that states no constraint searches the whole index. Chroma cannot search inside a joined value like `Drama|Thriller`, so each genre, country and director is stored as its own metadata flag. The filters cover only those four kinds of constraint, so a runtime limit is not applied at retrieval.

**Generation.** The second call sends the question and the numbered chunks, with instructions to answer only from them, cite their ids, avoid revealing endings and say "I don't know" otherwise. Citations are mapped back to the real chunk ids.

I ran four questions with `k = 5`, one of each kind and chosen before seeing any answers. The four cost $0.0012 in total (6,420 input and 1,080 output tokens) and took about 5 seconds each, apart from 15 seconds for the first, which includes loading the embedding model.

- **Question 1, a tense 1970s political thriller.** The year and genre filters were extracted and applied, and The Parallax View and Klute are grounded and cited. The third pick is the problem: All the President's Men is not in my catalog. The retrieved Klute chunk on themes mentions it as part of Pakula's "paranoia trilogy", so the model treated a film named in a source as a film in the Collection. A filter cannot prevent this, because it applies to the film a chunk belongs to and not to the films its text mentions.
- **Question 8, Wim Wenders films.** The director filter worked, and every retrieved chunk comes from one of his films. The answer still lists only 2 of the 10 films. The five chunks came from just 4 films, and the model left out Perfect Days and Buena Vista Social Club even though their chunks were retrieved. Five chunks cannot cover a complete list, even with the right filter.
- **Question 22, the plot of The Warped Ones.** The question states no hard constraint, so no filter applied. The system refused, which is honest, but this is a retrieval failure. The film's TMDB overview closely matches the question, yet none of its chunks reached the top five, which went to films with similar words such as Revenge.
- **Question 26, The Conformist.** No filter applied here either. It correctly said it does not know, because no retrieved chunk answers the question. The baseline answered this one from general knowledge, so this is the behavior I wanted from grounding.

**What this tells me.** The pipeline works end to end and cites its sources, the filters keep retrieval within the constraints a question states, and refusing when the sources are silent already fixes one baseline mistake. Four answers cannot show whether it beats the baseline, so the full evaluation still has to decide. Three weaknesses remain with the filters in place. Grounding does not keep films from outside the Collection out of the answers, because the sources mention them. Five chunks are too few for complete lists. Semantic search can also miss a specific plot description when the question gives no constraint to narrow it.

In [2]:
chunks = load_chunks(ROOT / "data" / "processed" / "chunks.jsonl")
collection = build_index(chunks)        # about 3 minutes the first time, then reused from models/chroma
print(f"{collection.count():,} chunks indexed")

27,576 chunks indexed


In [3]:
eval_set = pd.read_csv(ROOT / "data" / "evaluation_set.csv")
sample_ids = [1, 8, 22, 26]             # a recommendation, a complete list, a plot description, a film outside the Collection
sample_path = ROOT / "data" / "processed" / "pipeline_sample.json"

if sample_path.exists():                # saved run: delete the file to call the model again
    sample = json.loads(sample_path.read_text(encoding="utf-8"))
else:
    sample = [answer_question(eval_set.loc[eval_set["id"] == i, "question"].iloc[0], collection, k=5) for i in sample_ids]
    sample_path.write_text(json.dumps(sample, ensure_ascii=False, indent=1), encoding="utf-8")

for i, result in zip(sample_ids, sample):
    by_id = {chunk["id"]: chunk["metadata"] for chunk in result["retrieved"]}
    print("=" * 100)
    print(f"Question {i}: {result['question']}")
    print(f"Filters: {result['filters'] or 'none'}")
    print(f"\nAnswer: {result['answer']}")
    print(f"Picks: {result['picks'] or 'none'}   Refused: {result['refused']}")
    print("\nCited:")
    for chunk_id in result["cited_ids"]:
        print(f"  {format_citation(by_id[chunk_id])['label']}")
    print("\nRetrieved:")
    for chunk in result["retrieved"]:
        meta = chunk["metadata"]
        print(f"  {chunk['distance']:.3f}  {meta['title']} | {meta['section']}")
    print(f"\nTokens: {result['input_tokens']:,} in, {result['output_tokens']:,} out   "
          f"Cost: ${result['cost_usd']:.5f}   Time: {result['seconds']:.1f} s")

Question 1: What should I watch if I want a tense, 1970s political thriller?
Filters: {'year_min': 1970, 'year_max': 1979, 'genre': 'Thriller'}

Answer: Try The Parallax View for a political conspiracy thriller, with All the President's Men and Klute as related paranoid 1970s picks.
Picks: ['The Parallax View', "All the President's Men", 'Klute']   Refused: False

Cited:
  Klute (1971): Wikipedia, "Themes" (revision 1379290746)
  The Parallax View (1974): TMDB
  Klute (1971): Wikipedia, "Lead" (revision 1379290746)

Retrieved:
  1.041  Klute | Themes
  1.094  The Parallax View | TMDB overview
  1.112  Straw Dogs | Critical response
  1.148  Klute | Lead
  1.157  Don't Look Now | Influence

Tokens: 1,440 in, 517 out   Cost: $0.00040   Time: 15.3 s
Question 8: Which Wim Wenders films are in the Collection?
Filters: {'director': 'Wim Wenders'}

Answer: The Wim Wenders films in the Collection are Until the End of the World and Wings of Desire.
Picks: ['Until the End of the World', 'Wings o

---
## 2. One design choice, compared

### The catalog filter

**Why it comes first.** The most expensive mistake in my framing is recommending a film that is not in the Collection. All my chunks come from Collection films, but their text mentions other films, so neither grounding nor metadata filters can stop this. Section 1 showed it directly with All the President's Men.

**What I added.** The index metadata holds every title in the Collection, 1,915 in total, so each film the model lists as a pick is checked against that list. If a film is outside it, the model answers again with that film excluded, and any outside film that remains is removed. The filter is part of every version below, so the comparison in this section changes only `k`.

In [6]:
cat = prepare_catalogue(load_tables()[0])
catalog = catalog_titles(collection)      # every film title in the Collection, from the index metadata
frozen = freeze_filters(eval_set)         # the same extracted filters for every k, so only k changes
print(f"{len(catalog):,} titles in the catalog filter")

1,915 titles in the catalog filter


In [5]:
runs, scored = {}, {}
for k in (5, 10, 20, 40):
    name = f"catalog_k{k}"
    runs[k] = run_version(name, eval_set, collection, k, frozen, catalog=catalog)      # saved and reused
    scored[k] = evaluate(runs[k], eval_set, cat, name)

comparison = pd.DataFrame({k: version_summary(df, eval_set) for k, df in scored.items()}).T
comparison.index.name = "k"
comparison.round(4)

,retrieval_hit,list_coverage,citation_hit,auto_pass,pick_precision,titles_named,titles_outside,false_refusal,refused_unanswerable,total_cost_usd,mean_seconds,input_tokens
k,,,,,,,,,,,,
5,0.7059,0.2167,0.7059,0.32,0.4848,20.0,0.0,0.36,1.0,0.0083,5.2063,45500.0
10,0.7647,0.4083,0.7059,0.40,0.6364,25.0,0.0,0.28,1.0,0.0124,4.7203,77004.0
20,0.7647,0.5583,0.7059,0.48,0.7576,36.0,0.0,0.24,1.0,0.0184,4.8237,134974.0
40,1.0000,0.6333,0.8235,0.64,0.8485,43.0,0.0,0.16,1.0,0.0308,4.9983,255526.0


In [6]:
# what the catalog filter removed in each run
pd.DataFrame([{"k": k, "id": r["id"], "titles_removed": " | ".join(r["titles_removed"]), "picks_kept": " | ".join(r["picks"])}
              for k, results in runs.items() for r in results if r["titles_removed"]])

,k,id,titles_removed,picks_kept
0,5,1,All the President's Men,The Parallax View
1,10,1,All the President's Men,The Parallax View
2,10,5,The Magnificent Seven,Throne of Blood | Yojimbo | Sanjuro
3,20,5,The Magnificent Seven,Throne of Blood | Yojimbo | Sanjuro
4,40,5,The Magnificent Seven,Throne of Blood | Yojimbo | Kill!


In [7]:
# automatic pass or fail for each answerable question at each k
by_question = pd.DataFrame({k: df.set_index("id")["auto_pass"] for k, df in scored.items()})
by_question = by_question.join(eval_set.set_index("id")[["group", "answerable"]])
by_question[by_question["answerable"] == 1].drop(columns="answerable")

,5,10,20,40,group
id,,,,,
1,False,False,True,True,recommendation
2,False,False,True,True,recommendation
3,False,False,True,True,recommendation
4,False,False,False,False,recommendation
5,True,True,True,True,similarity
6,True,True,True,True,programming
7,False,False,False,False,catalog set
8,False,False,False,False,catalog set
9,False,False,False,False,catalog set


### Comparing `k`

**What I changed.** I compared `k` = 5, 10, 20 and 40. Everything else stayed fixed: the `all-MiniLM-L6-v2` index, the chunks, the prompts, the catalog filter, and the metadata filters, which were extracted once and reused so that every version retrieved under the same constraints. I fixed the selection rule before running these versions: highest retrieval hit rate, then list coverage, then automatic pass rate, then lowest cost.

| Version | Retrieval hit | List coverage | Automatic pass | Precision | Citation hit | Titles outside catalog | False refusals | Unanswerable refused | Cost of run |
|---|---|---|---|---|---|---|---|---|---|
| LLM alone (baseline) | n/a | n/a | 11 of 25 | 0.70 | 0 of 17 | 8 of 41 | 3 of 25 | 4 of 5 | $0.0098 |
| `k` = 5 | 12 of 17 | 0.22 | 8 of 25 | 0.48 | 12 of 17 | 0 of 20 | 9 of 25 | 5 of 5 | $0.0083 |
| `k` = 10 | 13 of 17 | 0.41 | 10 of 25 | 0.64 | 12 of 17 | 0 of 25 | 7 of 25 | 5 of 5 | $0.0124 |
| `k` = 20 | 13 of 17 | 0.56 | 12 of 25 | 0.76 | 12 of 17 | 0 of 36 | 6 of 25 | 5 of 5 | $0.0184 |
| **`k` = 40** | **17 of 17** | **0.63** | **16 of 25** | **0.85** | **14 of 17** | **0 of 43** | **4 of 25** | **5 of 5** | $0.0308 |

Retrieval hit counts a question when at least one retrieved chunk belongs to an expected film or director, and 17 of the answerable questions have expected sources. List coverage is the average share of required films that reach the retrieved chunks on questions 7, 8 and 15. Each run answers all 30 questions and includes the cost of extracting the metadata filters.

**What the catalog filter did.** None of the 124 titles named across the four runs is outside the catalog. The filter stepped in five times: it removed All the President's Men from question 1 at `k` = 5 and 10, and The Magnificent Seven from question 5 at `k` = 10, 20 and 40. It prevents the wrong pick but does not find a replacement, so at `k` = 5 and 10 question 1 is left with only The Parallax View and fails its check.

**What the comparison shows.**

- **More chunks helped on almost every measure.** Going from 5 to 40 took automatic passes from 8 to 16, precision from 0.48 to 0.85 and false refusals from 9 to 4.
- **The largest gain came at the last step.** From `k` = 20 to 40, questions 16, 18, 21 and 22 started passing and none stopped passing. The Warped Ones (22), which Section 1 could not find, only reaches the retrieved chunks at `k` = 40.
- **The extra cost is small.** A run of 30 questions goes from $0.0083 to $0.0308, about a tenth of a cent per question, and the average time stays near 5 seconds.

**Which version I keep.** My selection rule picks **`k` = 40 with the catalog filter**. It has the highest retrieval hit rate and list coverage, and also the most automatic passes, the highest precision and the fewest false refusals of the four. Against the baseline it passes 16 questions instead of 11, raises precision from 0.70 to 0.85, cites an expected source on 14 of 17 questions instead of none, names no film outside the catalog against 8 of 41, and refuses all five unanswerable questions. It costs about three times as much as the baseline, and its 4 false refusals are one more than the baseline's 3, so it narrowly misses the goal I set in notebook 01 of not giving up the low false-refusal rate.

**Limits of this result.** Each version was run once and the model's answers vary, so a one-question gap is within noise: question 18 passes at `k` = 10 and 40 but not at 20, and question 19 passes only at 10. `k` = 40 is the largest value I tested and it was the best, so a larger `k` might do better still. The retrieval hit rate is generous, because one expected film is enough: question 9 counts as a hit with only one of its three films retrieved. The catalog filter covers only the films the model lists as picks, and a Collection film named under an unusual alternate title could be removed by mistake.

**What `k` does not fix.** Questions 4, 7, 8, 9, 11, 14, 15 and 25 failed at every `k`, and question 19 passed only at `k` = 10. At `k` = 40:

- **Runtime (4):** the metadata filters have no runtime limit, and the system refused because the chunks it read do not give the runtime.
- **Complete lists (7, 8):** the retrieved chunks cover 7 of the 8 required Brazil films and 9 of the 10 Wim Wenders films, but the model names only 3 and 6. The Brazil filter also excludes Notorious, which qualifies only through its article.
- **Scattered facts (9, 11, 25):** only one of the three November 2026 titles was retrieved, the answer to question 11 calls Citizen Kane the first release without mentioning Grand Illusion, and the system refused the hypnosis list.
- **Counting and two steps (14, 15):** the answer to question 14 is 25, the number of films in the retrieved chunks, against a true count of 34. Question 15 needs the director found from the spine number before it can be used as a filter, so a tool would serve both better than more chunks.

---
## 3. The final version

Which version you will evaluate in notebook 03 and use in the app, and why. Save it: the index and the configuration, in `models/`.